# Three QUBO sectors, with the instances behind the published rows

Logistics, manufacturing and traffic, as published on
[zksf.org/applications](https://zksf.org/applications/). Each instance is built
here exactly as it was measured, solved exhaustively for its true optimum, then
run through QAOA on a local simulator.

Nothing here needs an account, a token or a clone: the instances are generated
from their seeds or embedded as literals. The one line that sends the same
circuit to a quantum processor is shown at the end and is not executed.

| Sector | Qubits | Exhaustive optimum | Published hardware row |
|---|---|---|---|
| Logistics | 16 | -292.3588 | Rigetti -216.06, gap 76.30 |
| Manufacturing | 16 | -351.2615 | Rigetti -346.31, gap 4.95 |
| Traffic | 30 | congestion 11 | Rigetti 0 of 500 valid assignments |


In [ ]:
!pip install -q qiskit qiskit-aer scipy numpy

## The shared parts

One QAOA builder and one scoring rule, used by all three sectors.

Scoring is by the QUBO's own value, penalties included, rather than by filtering
samples down to feasible ones. A 4x4 assignment problem has 24 feasible states
in 65,536, so a feasibility filter returns nothing from 500 shots and says
nothing about the run. The penalty terms already price infeasibility, which is
what putting them in the QUBO was for.


In [ ]:
import itertools
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

SEED = 20260919
sim = AerSimulator(seed_simulator=SEED)   # seeded: a top-to-bottom run reprints the same numbers


def qaoa_circuit(Q, params, p):
    """QAOA for a QUBO: rzz couplings and rz fields from Q, an rx mixer."""
    n = Q.shape[0]
    Qs = (Q + Q.T) / 2
    qc = QuantumCircuit(n)
    qc.h(range(n))
    h = [-Qs[i, i] / 2 - sum(Qs[i, j] for j in range(n) if j != i) / 4 for i in range(n)]
    for g, b in zip(params[:p], params[p:]):
        for i in range(n):
            for j in range(i + 1, n):
                if abs(Qs[i, j]) > 1e-12:
                    qc.rzz(2 * g * Qs[i, j] / 4, i, j)
            if abs(h[i]) > 1e-12:
                qc.rz(2 * g * h[i], i)
        for i in range(n):
            qc.rx(2 * b, i)
    qc.measure_all()
    return qc


def bits_to_x(bits):
    return np.array([int(c) for c in reversed(bits.replace(" ", ""))], int)


def energy(Q, x):
    return float(x @ Q @ x)


def best_energy(counts, Q):
    """The best QUBO value among the measured bitstrings."""
    return min(energy(Q, bits_to_x(b)) for b in counts)


def exhaustive_minimum(Q):
    n = Q.shape[0]
    return min(energy(Q, np.array(x, int)) for x in itertools.product([0, 1], repeat=n))


def run_qaoa(Q, p=2, restarts=3, maxiter=150, shots=512, seed=SEED):
    """COBYLA over the QAOA angles, best of several restarts."""
    rng = np.random.default_rng(seed)

    def sampled(params):
        return best_energy(sim.run(qaoa_circuit(Q, params, p), shots=shots).result().get_counts(), Q)

    best_params, best_value = None, np.inf
    for r in range(restarts):
        x0 = np.r_[np.full(p, 0.6), np.full(p, 0.4)] if r == 0 else rng.uniform(0, np.pi, 2 * p)
        res = minimize(sampled, x0, method="COBYLA", options={"maxiter": maxiter, "rhobeg": 0.4})
        v = sampled(res.x)
        if v < best_value:
            best_params, best_value = res.x, v
    return best_params, best_value

## 1. Logistics: 4 vehicles, 4 stops, seed 20260919

One binary per vehicle-stop pair, so 16 qubits. The cost is the distance a
vehicle drives to its stop. One stop per vehicle and one vehicle per stop are
penalty terms, weighted at four times the largest distance so that breaking
either one is never worth the saving.


In [ ]:
def logistics_qubo(seed=20260919):
    rng = np.random.default_rng(seed)
    v = s = 4
    depots = rng.uniform(0, 10, (v, 2))
    stops = rng.uniform(0, 10, (s, 2))
    dist = np.linalg.norm(depots[:, None, :] - stops[None, :, :], axis=2)

    pen = float(dist.max()) * 4.0
    Q = np.zeros((v * s, v * s))
    at = lambda a, b: a * s + b
    for a in range(v):
        for b in range(s):
            Q[at(a, b), at(a, b)] = float(dist[a, b]) - 2 * pen
    for a in range(v):                          # one stop per vehicle
        for b1, b2 in itertools.combinations(range(s), 2):
            Q[at(a, b1), at(a, b2)] += pen
            Q[at(a, b2), at(a, b1)] += pen
    for b in range(s):                          # one vehicle per stop
        for a1, a2 in itertools.combinations(range(v), 2):
            Q[at(a1, b), at(a2, b)] += pen
            Q[at(a2, b), at(a1, b)] += pen
    return Q


Q_log = logistics_qubo()
exact_log = exhaustive_minimum(Q_log)
print(f"logistics: {Q_log.shape[0]} qubits, exhaustive minimum {exact_log:.4f}")
# logistics: 16 qubits, exhaustive minimum -292.3588

In [ ]:
params_log, best_log = run_qaoa(Q_log)
print(f"QAOA p=2 on a simulator: {best_log:.4f}, gap {best_log - exact_log:.4f}")
print(f"published Rigetti row:   -216.0600, gap {-216.06 - exact_log:.4f}, 500 shots, $0.5125")

## 2. Manufacturing: 4 jobs, 4 time slots, seed 20260920

Also 16 qubits, one binary per job-slot pair. The cost is each job's weighted
lateness against its due slot. One slot per job is a penalty, and so is putting
two jobs that need the same machine into the same slot, which is what makes this
harder than a plain assignment.


In [ ]:
def manufacturing_qubo(seed=20260920):
    rng = np.random.default_rng(seed)
    j = t = 4
    due = rng.integers(0, t, j)
    machine = rng.integers(0, 2, j)             # two machines
    weight = rng.uniform(1, 4, j)
    lateness = np.array([[weight[a] * max(0, b - due[a]) for b in range(t)] for a in range(j)])

    pen = float(lateness.max() + 1) * 4.0
    Q = np.zeros((j * t, j * t))
    at = lambda a, b: a * t + b
    for a in range(j):
        for b in range(t):
            Q[at(a, b), at(a, b)] = float(lateness[a, b]) - 2 * pen
    for a in range(j):                          # one slot per job
        for b1, b2 in itertools.combinations(range(t), 2):
            Q[at(a, b1), at(a, b2)] += pen
            Q[at(a, b2), at(a, b1)] += pen
    for b in range(t):                          # no machine clash inside a slot
        for a1, a2 in itertools.combinations(range(j), 2):
            if machine[a1] == machine[a2]:
                Q[at(a1, b), at(a2, b)] += pen
                Q[at(a2, b), at(a1, b)] += pen
    return Q


Q_man = manufacturing_qubo()
exact_man = exhaustive_minimum(Q_man)
print(f"manufacturing: {Q_man.shape[0]} qubits, exhaustive minimum {exact_man:.4f}")
# manufacturing: 16 qubits, exhaustive minimum -351.2615

In [ ]:
params_man, best_man = run_qaoa(Q_man)
print(f"QAOA p=2 on a simulator: {best_man:.4f}, gap {best_man - exact_man:.4f}")
print(f"published Rigetti row:   -346.3100, gap {-346.31 - exact_man:.4f}, 500 shots, $0.5125")

## 3. Traffic: 10 vehicles, 3 routes each, seed 20260910

One binary per vehicle-route pair, so 30 qubits. One route per vehicle is a
penalty; every road segment two routes share becomes a coupling weighted by how
many segments they share. That gives the 228 two-qubit terms the page quotes,
and an exhaustive optimum of congestion 11.

Thirty qubits is past a local statevector, so the angles are optimised on the
24-qubit instance from the same generator and transferred, which is what the
page states and what the measured row used. Both fleets are embedded below:
`fleet[vehicle][route]` is the list of road segment ids that route uses.


In [ ]:
FLEET_24 = [[[1, 3, 7, 11], [9, 11], [2, 3, 11]], [[5, 9, 10], [2, 6, 9, 11], [0, 2, 4]], [[5, 6], [1, 7], [1, 2, 9]], [[2, 3, 7], [1, 4, 10], [4, 9, 11]], [[1, 4, 9, 10], [0, 10], [0, 5]], [[2, 8, 11], [1, 2], [4, 6, 7]], [[1, 3, 5, 8], [4, 5], [0, 1, 7]], [[2, 6, 8, 9], [4, 5, 6], [0, 2, 5, 8]]]

FLEET_30 = [[[5, 9, 10], [2, 3, 5, 9], [9, 10, 11, 13]], [[0, 2, 6], [0, 2, 6, 7], [3, 4, 5, 10]], [[5, 6], [11, 12], [3, 5, 9, 10]], [[4, 7, 12], [5, 13], [4, 10, 13]], [[4, 8, 11], [1, 2, 6, 8], [1, 11]], [[2, 12, 13], [0, 6, 12], [1, 2, 7, 8]], [[6, 11, 12], [1, 6], [3, 5]], [[0, 1, 10], [1, 3, 11], [3, 4, 12]], [[2, 4, 6], [10, 11, 13], [0, 6]], [[0, 4, 10], [5, 6, 8], [5, 8]]]


def traffic_qubo(fleet, routes=3, pen=4.0):
    v = len(fleet)
    n = v * routes
    at = lambda a, b: a * routes + b
    segs = [list(fleet[a][b]) for a in range(v) for b in range(routes)]

    Q = np.zeros((n, n))
    for a in range(v):                          # one route per vehicle
        for b in range(routes):
            Q[at(a, b), at(a, b)] -= 2 * pen
        for b1, b2 in itertools.combinations(range(routes), 2):
            Q[at(a, b1), at(a, b2)] += pen
            Q[at(a, b2), at(a, b1)] += pen
    for i, k in itertools.combinations(range(n), 2):
        if i // routes == k // routes:
            continue
        shared = len(set(segs[i]) & set(segs[k]))
        if shared:
            Q[i, k] += shared
            Q[k, i] += shared
    return Q, segs


def congestion(x, segs):
    """Vehicles past the first on a segment, summed: the objective the QUBO stands in for."""
    load = {}
    for k, on in enumerate(x):
        if on:
            for s in segs[k]:
                load[s] = load.get(s, 0) + 1
    return sum(max(0, c - 1) for c in load.values())


def one_route_each(x, routes=3):
    return bool((x.reshape(-1, routes).sum(axis=1) == 1).all())


Q_24, segs_24 = traffic_qubo(FLEET_24)
Q_30, segs_30 = traffic_qubo(FLEET_30)
for name, Q in (("24-qubit", Q_24), ("30-qubit", Q_30)):
    n = Q.shape[0]
    pairs = sum(1 for i, k in itertools.combinations(range(n), 2) if abs(Q[i, k]) > 1e-12)
    print(f"traffic {name} instance: {n} qubits, {pairs} couplings")
# traffic 24-qubit instance: 24 qubits, 167 couplings
# traffic 30-qubit instance: 30 qubits, 228 couplings

Optimise the angles on the 24-qubit instance. About a minute: each evaluation is
a 24-qubit statevector, roughly a second, and COBYLA takes 60 steps.

This leg scores by congestion over the samples that assign one route per
vehicle, rather than by QUBO energy, because congestion is the quantity the
traffic page reports. A draw with no such sample scores at the sentinel, which
is what keeps COBYLA moving away from angles that produce none.


In [ ]:
SENTINEL = 1e6


def survey(params, Q, segs, shots=512):
    """How many shots assigned one route per vehicle, and the best congestion among them."""
    counts = sim.run(qaoa_circuit(Q, params, 1), shots=shots).result().get_counts()
    valid, best = 0, np.inf
    for bits, k in counts.items():
        x = bits_to_x(bits)
        if one_route_each(x):
            valid += k
            best = min(best, congestion(x, segs))
    return valid, best


def sampled_congestion(params):
    _, best = survey(params, Q_24, segs_24)
    return best if np.isfinite(best) else SENTINEL


res = minimize(sampled_congestion, np.array([0.6, 0.4]), method="COBYLA",
               options={"maxiter": 60, "rhobeg": 0.4})
valid, best = survey(res.x, Q_24, segs_24)
print(f"angles from the 24-qubit instance: {np.round(res.x, 4).tolist()}")
print(f"at those angles: {valid} of 512 shots assigned one route per vehicle")
print(f"best congestion among them: {'none in this draw' if not np.isfinite(best) else int(best)}"
      f", against an exhaustive optimum of 8")

The optimiser settles on gamma 1.0, beta 0.4, which are the angles the published
30-qubit row was run with, and the draw at those angles contains no assignment
that gives every vehicle exactly one route. That is the noiseless simulator: at
depth 1 and 512 shots, this width produces none. The hardware row returning 0 of
500 is the same effect rather than a device fault, and the fix is depth or a
constraint-preserving mixer rather than a better processor.


In [ ]:
# The 30-qubit circuit those angles are carried to. It is built, not simulated:
# 30 qubits is 2^30 amplitudes, and a matrix-product state does not help at 228
# couplings. This is the circuit the published Rigetti row ran.
qc_30 = qaoa_circuit(Q_30, res.x, 1)
print(f"depth {qc_30.depth()}, two-qubit gates {qc_30.count_ops().get('rzz', 0)}, "
      f"qubits {qc_30.num_qubits}")
print("published Rigetti row: 0 of 500 shots were valid assignments, $0.5125")
print("exhaustive optimum for this instance: congestion 11, settled in 0.18 s classically")

## Sending any of these to a quantum processor

One field. Not run here: it bills, and the result would be a fourth number
measured on a different day on a shared device rather than anything this
notebook can check.

```python
import qsim_sdk

client = qsim_sdk.Client(token="YOUR_TOKEN")

client.estimate(qc_30, shots=500, engine="qpu.rigetti")   # free, quotes the cost
job = client.run(qc_30, shots=500, engine="qpu.rigetti")  # bills on submit
best_energy(job.counts, Q_30)
```

At 500 shots: Rigetti $0.5125, IQM Garnet $1.025, IQM Emerald $1.10. Swap
`engine` for `exact.cpu`, `mps.quimb.cpu` or `exact.gpu` to run the identical
circuit on a simulator tier first, which is what the CPU and GPU rows on each
sector page are.

The step-by-step version of all of this, including the two modes (your own QUBO,
or these exact instances), is in the [docs](https://zksf.org/docs/#reproduce).
